In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yasa
import psutil
from psutil._common import bytes2human

from pathlib import Path
from scipy.io import loadmat
from mne.filter import resample
from itertools import chain
from tqdm import tqdm
from sys import getsizeof

from functions import get_sequences
from functions import phasic_rem_v3
from functions import create_name_cbd, create_name_rgs, create_name_os
from runtime_logger import logger_setup

CBD_DIR = "/home/nero/datasets/CBD/"
RGS_DIR = "/home/nero/datasets/RGS14/"
OS_DIR = "/home/nero/datasets/OSbasic/"

OUTPUT_DIR  = "/home/nero/datasets/preprocesssed/"
CBD_OVERVIEW_PATH = "overview.csv"

overview_df = pd.read_csv(CBD_OVERVIEW_PATH)

fs_cbd = 2500
fs_os = 2500
fs_rgs = 1000

targetFs = 500
n_down_cbd = fs_cbd/targetFs
n_down_rgs = fs_rgs/targetFs
n_down_os = fs_os/targetFs

phasic_tonic_idx = {}

phasic_tonic_freq = {
    'rat_id': [],
    'study_day': [],
    'condition': [],
    'treatment': [],
    'trial_num': [],
    'state': [],
    'count': [],
    'count_norm': [],
    'duration': []
    }

phasic_tonic_dur = {
    'rat_id': [],
    'study_day': [],
    'condition': [],
    'treatment': [],
    'trial_num': [],
    'state': [],
    'duration': [],
    }

logger = logger_setup()

# CBD Dataset

In [2]:
pattern = r"[\w-]+posttrial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(CBD_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of recordings: ", len(mapped))

Number of recordings:  170


In [3]:
fs = 2500

keys = iter(mapped.keys())
state = next(keys)
hpc = mapped[state]
title = create_name_cbd(hpc, overview_df)

# Load the LFP data
lfpHPC = loadmat(hpc)['HPC']
lfpHPC = lfpHPC.flatten()

# Load the states
hypno = loadmat(state)['states']
hypno = hypno.flatten()

lfpHPC.shape
print("Memory available: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
print("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))

data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")
print("Resampled data shape: {0}, size: {1}B)".format(str(data_resample.shape), getsizeof(data_resample)))

# Remove artifacts
art_std, zscores_std = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
data_resample[art_up] = 0

print("art_up shape: {0}, size: {1}B)".format(str(art_up.shape), getsizeof(art_up)))

data_resample = data_resample - data_resample.mean()

Memory available: 5.8G (23.9%)
LFP shape: (6752854,), size: 51MB)
Resampled data shape: (1350571,), size: 112B)
art_up shape: (1350571,), size: 1350683B)


In [4]:
from scipy.signal import hilbert
from neurodsp.filt import filter_signal

def get_sequences(idx, ibreak=1) :  
    """
    get_sequences(idx, ibreak=1)
    idx     -    np.vector of indices
    @RETURN:
    seq     -    list of np.vectors
    """
    diff = idx[1:] - idx[0:-1]
    breaks = np.nonzero(diff>ibreak)[0]
    breaks = np.append(breaks, len(idx)-1)
    
    seq = []    
    iold = 0
    for i in breaks:
        r = list(range(iold, i+1))
        seq.append(idx[r])
        iold = i+1
        
    return seq

def my_bpfilter(x, w0, w1, N=4,bf=True):
    """
    create N-th order bandpass Butterworth filter with corner frequencies 
    w0*sampling_rate/2 and w1*sampling_rate/2
    """
    #from scipy import signal
    #taps = signal.firwin(numtaps, w0, pass_zero=False)
    #y = signal.lfilter(taps, 1.0, x)
    #return y
    from scipy import signal
    b,a = signal.butter(N, [w0, w1], 'bandpass')
    if bf:
        y = signal.filtfilt(b,a, x)
    else:
        y = signal.lfilter(b,a, x)
        
    return y

def _detect_troughs(signal, thr):
    lidx  = np.where(signal[0:-2] > signal[1:-1])[0]
    ridx  = np.where(signal[1:-1] <= signal[2:])[0]
    thidx = np.where(signal[1:-1] < thr)[0]
    sidx = np.intersect1d(lidx, np.intersect1d(ridx, thidx))+1
    return sidx

def downsample_vec(x, nbin):
    """
    y = downsample_vec(x, nbin)
    downsample the vector x by replacing nbin consecutive \
    bin by their mean \
    @RETURN: the downsampled vector 
    """
    n_down = int(np.floor(len(x) / nbin))
    x = x[0:n_down*nbin]
    x_down = np.zeros((n_down,))

    # 0 1 2 | 3 4 5 | 6 7 8 
    for i in range(nbin) :
        idx = list(range(i, int(n_down*nbin), int(nbin)))
        x_down += x[idx]


# Parameters
nfilt = 11
min_dur = 2.5

In [5]:
seq = get_sequences(np.where(hypno == 5)[0]) # list of np.arrays 
seq

[array([1331, 1332, 1333, 1334, 1335, 1336, 1337, 1338, 1339, 1340, 1341,
        1342, 1343, 1344, 1345, 1346, 1347, 1348, 1349, 1350, 1351, 1352,
        1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1361, 1362, 1363,
        1364, 1365, 1366, 1367, 1368, 1369, 1370, 1371, 1372, 1373, 1374,
        1375, 1376, 1377, 1378, 1379, 1380, 1381, 1382, 1383, 1384, 1385,
        1386, 1387, 1388, 1389, 1390, 1391, 1392, 1393, 1394, 1395, 1396,
        1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1406, 1407,
        1408, 1409, 1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417, 1418,
        1419, 1420, 1421, 1422, 1423, 1424, 1425, 1426, 1427, 1428, 1429,
        1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1438, 1439, 1440,
        1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451,
        1452, 1453, 1454, 1455, 1456, 1457, 1458, 1459, 1460, 1461, 1462,
        1463, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1471, 1472, 1473,
        1474, 1475, 1476, 1477, 1478, 

In [25]:
neeg = EEG.shape[0] # length
seq = get_sequences(np.where(hypno == 5)[0]) # list of np.arrays 
rem_idx = []
for s in seq:
    rem_idx += list(s) # list of REM indices

a=[]
sr = targetFs
nbin = sr
sdt = nbin*(1/sr)

# Theta band pass filter freq
w1 = 5.0
w2 = 12.0
    
filt = np.ones((nfilt,))
filt = filt / filt.sum()

trdiff_list = []
tridx_list = []
rem_eeg = np.array([])
eeg_seq = {}
sdiff_seq = {}
tridx_seq = {}

# Collect for each REM sequence the smoothed inter-trough intervals
# and EEG amplitudes as well as the indices of the troughs.

seq = [s for s in seq if len(s)>=min_dur] # min duration of each REM epoch should be greater than 2.5 s

for s in seq:
    ta = s[0]*nbin
    #tb = s[-1]*(nbin+1)
    tb = (s[-1]+1)*nbin
    tb = np.min((tb, neeg))
                
    eeg_idx = np.arange(ta, tb) # indices for the REM epoch      
    eeg = EEG[eeg_idx] # rem epoch 
    if len(eeg)*(1/sr) <= min_dur: # already checked before
        continue
    eegh =  filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
    res = hilbert(eegh)
    instantaneous_phase = np.angle(res)
    amp = np.abs(res)

    # trough indices
    tridx = _detect_troughs(instantaneous_phase, -3)
    # Alternative that does not seems to work that well:        
    #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
    
    # differences between troughs
    trdiff = np.diff(tridx)
   
    # smoothed trough differences
    sdiff_seq[s[0]] = np.convolve(trdiff, filt, 'same')
    # dict of trough differences for each REM period
    tridx_seq[s[0]] = tridx
    
    eeg_seq[s[0]] = amp

# Save smoothed intervals, trough differences and amplitude

In [28]:
sdiff_seq

{1331: array([36.63636364, 41.72727273, 49.36363636, ..., 47.63636364,
        41.81818182, 35.45454545]),
 2428: array([30.72727273, 37.27272727, 44.        , 48.54545455, 53.63636364,
        63.54545455, 64.27272727, 64.45454545, 65.72727273, 63.        ,
        62.54545455, 62.72727273, 60.90909091, 59.54545455, 60.18181818,
        59.63636364, 55.45454545, 54.36363636, 54.72727273, 54.63636364,
        55.09090909, 54.36363636, 54.        , 53.90909091, 52.63636364,
        53.45454545, 54.81818182, 52.18181818, 53.        , 57.36363636,
        58.81818182, 58.27272727, 58.27272727, 56.54545455, 58.18181818,
        59.09090909, 58.        , 58.54545455, 61.45454545, 63.09090909,
        60.18181818, 60.        , 61.45454545, 62.18181818, 66.36363636,
        65.90909091, 66.54545455, 67.45454545, 67.45454545, 68.81818182,
        67.09090909, 66.54545455, 68.36363636, 67.36363636, 66.09090909,
        63.27272727, 61.63636364, 60.72727273, 63.72727273, 62.72727273,
        60.

In [29]:
# collect again smoothed inter-trough differences and amplitude;
# but this time concat the data to one long vector each (@trdiff_sm and rem_eeg)
for s in seq:
    ta = s[0]*nbin
    tb = (s[-1]+1)*nbin
    tb = np.min((tb, neeg))
    eeg_idx = np.arange(ta, tb)
    eeg = EEG[eeg_idx]            
    if len(eeg)*(1/sr) <= min_dur:
        continue
    
    eegh = filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
    res = hilbert(eegh)
    instantaneous_phase = np.angle(res)
    amp = np.abs(res)

    # trough indices
    tridx = _detect_troughs(instantaneous_phase, -3)
    # alternative version:
    #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
    
    # differences between troughs
    tridx_list.append(tridx+ta)
    trdiff = np.diff(tridx)
    trdiff_list += list(trdiff)
   
    rem_eeg = np.concatenate((rem_eeg, amp)) # concatenated instantaneous amplitude

trdiff = np.array(trdiff_list)
trdiff_sm = np.convolve(trdiff, filt, 'same')

# potential candidates for phasic REM:
# the smoothed difference between troughs is less than
# the 10th percentile:
thr1 = np.percentile(trdiff_sm, 10)
# the minimum difference in the candidate phREM is less than
# the 5th percentile
thr2 = np.percentile(trdiff_sm, 5)
# the peak amplitude is larger than the mean of the amplitude
# of the REM EEG.
thr3 = rem_eeg.mean()


In [ ]:
# 1. Downsample => Remove artifact => Extract REM epoch into { (start, end) : REM epoch signal}
# 2. REM epoch => Bandpass => Hilbert => Detect troughs => Save smoothed troughs & amplitude


In [ ]:
def detect_phasic_rem(epoch, indices, sr, min_dur=2.5, thr_dur = 900, nfilt=11):
        neeg = EEG.shape[0]
        seq = get_sequences(np.where(M == 5)[0])
        rem_idx = []
        for s in seq:
            rem_idx += list(s)
        
        sr = sr
        nbin = sr
        sdt = nbin*(1/sr)
        
        w1 = 5.0
        w2 = 12.0
        
        filt = np.ones((nfilt,))
        filt = filt / filt.sum()
        
        trdiff_list = []
        tridx_list = []
        rem_eeg = np.array([])
        eeg_seq = {}
        sdiff_seq = {}
        tridx_seq = {}
        
        # Collect for each REM sequence the smoothed inter-trough intervals
        # and EEG amplitudes as well as the indices of the troughs.
        seq = [s for s in seq if len(s)>=min_dur]
        for s in seq:
            ta = s[0]*nbin
            #tb = s[-1]*(nbin+1)
            tb = (s[-1]+1)*nbin
            tb = np.min((tb, neeg))
                    
            eeg_idx = np.arange(ta, tb) # this the whole REM epoch       
            eeg = EEG[eeg_idx]    
            if len(eeg)*(1/sr) <= min_dur:
                continue

            eegh =  filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
            res = hilbert(eegh)
            instantaneous_phase = np.angle(res)
            amp = np.abs(res)
        
            # trough indices
            tridx = _detect_troughs(instantaneous_phase, -3)
            # Alternative that does not seems to work that well:        
            #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1
            
            # differences between troughs
            trdiff = np.diff(tridx)
        
            # smoothed trough differences
            sdiff_seq[s[0]] = np.convolve(trdiff, filt, 'same')

            # dict of trough differences for each REM period
            tridx_seq[s[0]] = tridx
            
            eeg_seq[s[0]] = amp
        
        rem_idx = []    
        for s in seq:
            rem_idx += list(s)
        


        # collect again smoothed inter-trough differences and amplitude;
        # but this time concat the data to one long vector each (@trdiff_sm and rem_eeg)
        for s in seq:
            ta = s[0]*nbin
            tb = (s[-1]+1)*nbin
            tb = np.min((tb, neeg))

            eeg_idx = np.arange(ta, tb)
            eeg = EEG[eeg_idx]            
            if len(eeg)*(1/sr) <= min_dur:
                continue
            
            eegh = filter_signal(eeg, sr, 'bandpass',(w1,w2), remove_edges=False)
            res = hilbert(eegh)
            instantaneous_phase = np.angle(res)
            amp = np.abs(res)
        
            # trough indices
            tridx = _detect_troughs(instantaneous_phase, -3)
            # alternative version:
            #tridx = np.where(np.diff(np.sign(np.diff(eegh))))[0]+1

            # differences between troughs
            tridx_list.append(tridx+ta)
            trdiff = np.diff(tridx)
            trdiff_list += list(trdiff)
        
            rem_eeg = np.concatenate((rem_eeg, amp)) 
        
        trdiff = np.array(trdiff_list)
        trdiff_sm = np.convolve(trdiff, filt, 'same')

        # potential candidates for phasic REM:
        # the smoothed difference between troughs is less than
        # the 10th percentile:
        thr1 = np.percentile(trdiff_sm, 10)
        # the minimum difference in the candidate phREM is less than
        # the 5th percentile
        thr2 = np.percentile(trdiff_sm, 5)
        # the peak amplitude is larger than the mean of the amplitude
        # of the REM EEG.
        thr3 = rem_eeg.mean()

        phrem = {}
        for si in tridx_seq:
            offset = nbin*si
            
            tridx = tridx_seq[si]
            sdiff = sdiff_seq[si]
            eegh = eeg_seq[si]
            
            idx = np.where(sdiff <= thr1)[0]
            cand = get_sequences(idx)
        
            #thr4 = np.mean(eegh)    
            for q in cand:
                dur = ( (tridx[q[-1]]-tridx[q[0]]+1)/sr ) * 1000
                #if 16250 > si*nbin * (1/sr) > 16100:
                #    print((tridx[q[0]]+si*nbin) * (1/sr))

                if dur > thr_dur and np.min(sdiff[q]) < thr2 and np.mean(eegh[tridx[q[0]]:tridx[q[-1]]+1]) > thr3:
                    
                    a = tridx[q[0]]   + offset
                    b = tridx[q[-1]]  + offset
                    idx = (a,b)
        
                    if si in phrem:
                        phrem[si].append(idx)
                    else:
                        phrem[si] = [idx]
        return phrem

In [3]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_cbd(hpc, overview_df)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        fname = OUTPUT_DIR + title
        np.savez(fname, data_resample)
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        logger.debug("Saving downsampled data ({0}) to {1}.".format(str(data_resample.shape), title))

        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)

            
            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################
        

  0%|          | 0/170 [00:01<?, ?it/s, Rat5_SD8_HC_0_posttrial3]


# RGS14 Dataset

In [4]:
pattern1 = r"[\w-]+post[\w-]+trial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(RGS_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern1, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))

Number of trials:  166


In [5]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_rgs(hpc)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_rgs, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        fname = OUTPUT_DIR + title
        np.savez(fname, data_resample)
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        logger.debug("Saving downsampled data ({0}) to {1}.".format(str(data_resample.shape), title))

        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)

            
            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################
        

  0%|          | 0/166 [00:00<?, ?it/s, Rat2_SD3_CON_2_posttrial1]

  0%|          | 0/166 [00:00<?, ?it/s, Rat2_SD3_CON_2_posttrial1]


# OSbasic Dataset

In [6]:
pattern2 = r".*post_trial.*"
mapped = {}

for root, dirs, fils in os.walk(OS_DIR):
    for dir in dirs:
        dir = (os.path.join(root, dir))
        # Check if the directory is a post trial directory
        if re.match(pattern2, dir, flags=re.IGNORECASE):
            dir = Path(dir)
            HPC_file = next(dir.glob("*HPC*"))
            states = next(dir.glob('*states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))


Number of trials:  210


In [7]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_os(hpc)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_os, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        fname = OUTPUT_DIR + title
        np.savez(fname, data_resample)
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        logger.debug("Saving downsampled data ({0}) to {1}.".format(str(data_resample.shape), title))

        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)

            
            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################
        

  0%|          | 0/210 [00:00<?, ?it/s, Rat1_SD1_OD_4_posttrial5]

  0%|          | 0/210 [00:25<?, ?it/s, Rat1_SD1_OD_4_posttrial5]


# Save

In [8]:
np.save('phasic_tonic_idx', phasic_tonic_idx)
# For loading use
# data = np.load('data.npy', allow_pickle='TRUE').item()

ep_df = pd.DataFrame({key:pd.Series(value) for key, value in phasic_tonic_freq.items()})
ep_df.to_csv("states_freq_dur.csv", index=False)

dur_df = pd.DataFrame({key:pd.Series(value) for key, value in phasic_tonic_dur.items()})
dur_df.to_csv("states_durations.csv", index=False)

In [13]:
ep_df

,rat_id,study_day,condition,treatment,trial_num,state,count,count_norm,duration
0,5,8,HC,0,3,phasic,4,0.088889,10.270
1,5,8,HC,0,3,tonic,6,0.133333,190.730
2,2,3,CON,2,1,phasic,2,0.044444,2.470
3,2,3,CON,2,1,tonic,4,0.088889,109.530
4,1,1,OD,4,5,phasic,28,0.155556,54.548
5,1,1,OD,4,5,tonic,37,0.205556,1250.452


In [11]:
dur_df

,rat_id,study_day,condition,treatment,trial_num,state,duration
0,5,8,HC,0,3,phasic,1.784
1,5,8,HC,0,3,phasic,3.492
2,5,8,HC,0,3,phasic,2.626
3,5,8,HC,0,3,phasic,2.368
4,5,8,HC,0,3,tonic,48.982
...,...,...,...,...,...,...,...
76,1,1,OD,4,5,tonic,17.536
77,1,1,OD,4,5,tonic,17.310
78,1,1,OD,4,5,tonic,55.784
79,1,1,OD,4,5,tonic,65.288
